# CWRU Color Spectrogram Model Training & Report Figures

이 노트북은 컬러 스펙트로그램 데이터셋을 학습하고, 보고서에 넣을 수 있는 비교 피규어를 자동으로 저장하는 파일이다.

포함 모델:

- SmallSpectroCNN
- ResNet18 from scratch
- ResNet18 fine-tuning
- MobileNetV3-Small
- EfficientNet-B0

생성 결과:

- train/validation accuracy 수렴 곡선
- train/validation/test accuracy 비교 그래프
- epoch당 학습 시간 비교 그래프
- 모델별 confusion matrix
- 최고 성능 모델 Grad-CAM 예시


In [1]:
"""
CWRU color spectrogram classification training script.

Recommended workflow:
1. Run 02_make_training_dataset_sliding_COLOR.ipynb first.
2. Check that this folder exists:
   ./cwru_spectrogram_training_dataset/split_dataset/train
3. Run this script.

This script trains and compares:
- SmallSpectroCNN
- ResNet18 from scratch
- ResNet18 fine-tuning (ImageNet weights if available)
- MobileNetV3-Small
- EfficientNet-B0

Outputs:
- trained_models/*.pth
- reports/history_all.csv
- reports/model_comparison.csv
- reports/*.png report figures
"""

from __future__ import annotations

import json
import random
import time
import copy
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import datasets, models

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x




In [2]:
# =========================
# User settings


In [3]:
# =========================
DATASET_ROOT = Path("./cwru_spectrogram_training_dataset/split_dataset")
OUTPUT_DIR = Path("./cwru_model_comparison_results")
MODEL_DIR = OUTPUT_DIR / "trained_models"
REPORT_DIR = OUTPUT_DIR / "reports"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 0  # Windows에서는 0이 가장 안전하다.

# 빠른 테스트는 아래 리스트를 줄이면 된다.
MODEL_NAMES = [
    "small_cnn",
    "resnet18_scratch",
    "resnet18_finetune",
    "mobilenet_v3_small",
    "efficientnet_b0",
]

# True이면 torchvision의 ImageNet pretrained weight를 사용하려고 시도한다.
# 인터넷/캐시 문제로 실패하면 자동으로 random init으로 fallback한다.
USE_PRETRAINED_IF_AVAILABLE = True



In [4]:
# =========================
# Reproducibility


In [5]:
# =========================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




In [6]:
# =========================
# Model definitions


In [7]:
# =========================
class SmallSpectroCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def _safe_weights(weights_enum):
    if not USE_PRETRAINED_IF_AVAILABLE:
        return None
    try:
        return weights_enum.DEFAULT
    except Exception:
        return None


def build_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()

    if name == "small_cnn":
        return SmallSpectroCNN(num_classes)

    if name == "resnet18_scratch":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if name == "resnet18_finetune":
        try:
            weights = _safe_weights(models.ResNet18_Weights)
            model = models.resnet18(weights=weights)
        except Exception:
            model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    if name == "mobilenet_v3_small":
        try:
            weights = _safe_weights(models.MobileNet_V3_Small_Weights)
            model = models.mobilenet_v3_small(weights=weights)
        except Exception:
            model = models.mobilenet_v3_small(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    if name == "efficientnet_b0":
        try:
            weights = _safe_weights(models.EfficientNet_B0_Weights)
            model = models.efficientnet_b0(weights=weights)
        except Exception:
            model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unknown model name: {name}")


def get_target_layer(model: nn.Module, model_name: str):
    if model_name == "small_cnn":
        return model.features[-2]  # last ReLU before GAP
    if model_name.startswith("resnet18"):
        return model.layer4[-1]
    if model_name == "mobilenet_v3_small":
        return model.features[-1]
    if model_name == "efficientnet_b0":
        return model.features[-1]
    return None




In [8]:
# =========================
# Data


In [9]:
# =========================
def build_dataloaders(dataset_root: Path):
    train_tf = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.RandomApply([T.ColorJitter(brightness=0.15, contrast=0.15)], p=0.3),
        T.RandomAffine(degrees=3, translate=(0.02, 0.02)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = datasets.ImageFolder(dataset_root / "train", transform=train_tf)
    val_ds = datasets.ImageFolder(dataset_root / "val", transform=eval_tf)
    test_ds = datasets.ImageFolder(dataset_root / "test", transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, val_loader, test_loader, train_ds.classes




In [10]:
# =========================
# Training and evaluation


In [11]:
# =========================
def run_one_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train(train)
    losses, ys, preds = [], [], []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()

        losses.append(loss.item() * images.size(0))
        ys.extend(labels.detach().cpu().numpy().tolist())
        preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    avg_loss = float(np.sum(losses) / len(loader.dataset))
    acc = accuracy_score(ys, preds)
    macro_f1 = f1_score(ys, preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    ys, preds = [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        ys.extend(labels.numpy().tolist())
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(ys), np.array(preds)


def train_model(model_name: str, train_loader, val_loader, test_loader, class_names, device):
    num_classes = len(class_names)
    model = build_model(model_name, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_state = None
    best_val_acc = -1.0
    history = []
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_one_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc, val_f1 = run_one_epoch(model, val_loader, criterion, optimizer, device, train=False)
        scheduler.step()

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(f"[{model_name}] epoch {epoch:03d}/{NUM_EPOCHS} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    elapsed = time.time() - start_time
    if best_state is not None:
        model.load_state_dict(best_state)

    y_test, p_test = predict_all(model, test_loader, device)
    test_acc = accuracy_score(y_test, p_test)
    test_f1 = f1_score(y_test, p_test, average="macro", zero_division=0)

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_name": model_name,
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "test_acc": test_acc,
        "test_macro_f1": test_f1,
    }, MODEL_DIR / f"{model_name}.pth")

    history_df = pd.DataFrame(history)

    summary = {
        "model": model_name,
        "final_train_acc": float(history_df["train_acc"].iloc[-1]),
        "best_train_acc": float(history_df["train_acc"].max()),
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "test_macro_f1": test_f1,
        "train_time_sec": elapsed,
        "sec_per_epoch": elapsed / NUM_EPOCHS,
        "num_epochs": NUM_EPOCHS,
    }
    return model, history_df, summary, y_test, p_test




In [12]:
# =========================
# Figures


In [19]:
# =========================
def save_curve_figures(history_df: pd.DataFrame):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    epoch_ticks = sorted(history_df["epoch"].unique())

    plt.figure(figsize=(9, 5))
    for model_name, g in history_df.groupby("model"):
        plt.plot(g["epoch"], g["train_acc"], marker="o", label=model_name)
    plt.xlabel("Epoch")
    plt.ylabel("Train Accuracy")
    plt.title("Train Accuracy Convergence")
    plt.xticks(epoch_ticks)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_train_accuracy_convergence.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 5))
    for model_name, g in history_df.groupby("model"):
        plt.plot(g["epoch"], g["val_acc"], marker="o", label=model_name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.title("Validation Accuracy Convergence")
    plt.xticks(epoch_ticks)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_val_accuracy_convergence.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 5))
    for model_name, g in history_df.groupby("model"):
        plt.plot(g["epoch"], g["train_acc"], marker="o", label=f"{model_name} train")
        plt.plot(g["epoch"], g["val_acc"], marker="s", linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Train and Validation Accuracy Convergence")
    plt.xticks(epoch_ticks)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_train_val_accuracy_convergence.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 5))
    for model_name, g in history_df.groupby("model"):
        plt.plot(g["epoch"], g["val_loss"], marker="o", label=model_name)
    plt.xlabel("Epoch")
    plt.ylabel("Validation Loss")
    plt.title("Validation Loss Convergence")
    plt.xticks(epoch_ticks)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_val_loss_convergence.png", dpi=200)
    plt.close()


def save_comparison_figures(summary_df: pd.DataFrame):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    x = np.arange(len(summary_df))
    labels = summary_df["model"].tolist()

    plt.figure(figsize=(10, 5))
    plt.bar(x, summary_df["final_train_acc"])
    plt.xticks(x, labels, rotation=25, ha="right")
    plt.ylabel("Final Train Accuracy")
    plt.title("Final Train Accuracy Comparison")
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_train_accuracy_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(x, summary_df["test_acc"])
    plt.xticks(x, labels, rotation=25, ha="right")
    plt.ylabel("Test Accuracy")
    plt.title("Final Test Accuracy Comparison")
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_test_accuracy_comparison.png", dpi=200)
    plt.close()

    width = 0.25
    plt.figure(figsize=(11, 5))
    plt.bar(x - width, summary_df["final_train_acc"], width, label="Train")
    plt.bar(x, summary_df["best_val_acc"], width, label="Validation")
    plt.bar(x + width, summary_df["test_acc"], width, label="Test")
    plt.xticks(x, labels, rotation=25, ha="right")
    plt.ylabel("Accuracy")
    plt.title("Train / Validation / Test Accuracy Comparison")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_train_val_test_accuracy_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.bar(x, summary_df["sec_per_epoch"])
    plt.xticks(x, labels, rotation=25, ha="right")
    plt.ylabel("Seconds per Epoch")
    plt.title("Training Speed Comparison")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "fig_training_speed_comparison.png", dpi=200)
    plt.close()


def save_confusion_matrix_figure(y_true, y_pred, class_names, model_name: str):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    plt.figure(figsize=(8, 7))
    plt.imshow(cm_norm, aspect="auto")
    plt.title(f"Normalized Confusion Matrix - {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.colorbar(label="Normalized count")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f"fig_confusion_matrix_{model_name}.png", dpi=200)
    plt.close()


In [14]:
# =========================
# Grad-CAM


In [15]:
# =========================
def denormalize_image(tensor_img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor_img.cpu() * std + mean
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def disable_inplace_activations(model):
    for module in model.modules():
        if hasattr(module, "inplace"):
            module.inplace = False
    return model


def make_gradcam(model, model_name, dataset, device, save_name="fig_gradcam_best_model.png"):
    target_layer = get_target_layer(model, model_name)
    if target_layer is None:
        return

    model = disable_inplace_activations(model)
    model.eval()
    image, label = dataset[0]
    input_tensor = image.unsqueeze(0).to(device)

    activations = {}
    gradients = {}

    def fwd_hook(module, inp, out):
        activations["value"] = out.detach()

    def bwd_hook(module, grad_in, grad_out):
        gradients["value"] = grad_out[0].detach()

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)

    logits = model(input_tensor)
    pred = logits.argmax(dim=1).item()
    score = logits[0, pred]
    model.zero_grad(set_to_none=True)
    score.backward()

    acts = activations["value"]
    grads = gradients["value"]
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = (weights * acts).sum(dim=1).squeeze(0)
    cam = torch.relu(cam)
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    cam = torch.nn.functional.interpolate(cam[None, None], size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
    cam = cam.squeeze().cpu().numpy()

    img = denormalize_image(image)

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.imshow(cam, alpha=0.45)
    plt.axis("off")
    plt.title(f"Grad-CAM: {model_name}\ntrue={dataset.classes[label]}, pred={dataset.classes[pred]}")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / save_name, dpi=200)
    plt.close()

    h1.remove()
    h2.remove()




In [16]:
# =========================
# Main


In [20]:
# =========================
def main():
    set_seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    if not (DATASET_ROOT / "train").exists():
        raise FileNotFoundError(
            f"{DATASET_ROOT / 'train'} 폴더가 없다. 먼저 컬러 스펙트로그램 데이터셋 생성 노트북을 실행해야 한다."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_loader, val_loader, test_loader, class_names = build_dataloaders(DATASET_ROOT)
    with open(OUTPUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
        json.dump(class_names, f, ensure_ascii=False, indent=2)

    all_histories = []
    summaries = []
    preds_by_model = {}
    trained_models = {}

    for model_name in MODEL_NAMES:
        model, hist_df, summary, y_test, p_test = train_model(
            model_name, train_loader, val_loader, test_loader, class_names, device
        )
        all_histories.append(hist_df)
        summaries.append(summary)
        preds_by_model[model_name] = (y_test, p_test)
        trained_models[model_name] = model
        save_confusion_matrix_figure(y_test, p_test, class_names, model_name)

    history_df = pd.concat(all_histories, ignore_index=True)
    summary_df = pd.DataFrame(summaries).sort_values("test_acc", ascending=False).reset_index(drop=True)

    history_df.to_csv(REPORT_DIR / "history_all.csv", index=False, encoding="utf-8-sig")
    summary_df.to_csv(REPORT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

    save_curve_figures(history_df)
    save_comparison_figures(summary_df)

    # Grad-CAM은 test accuracy 기준 최고 모델 하나만 생성한다.
    best_name = summary_df.loc[0, "model"]
    _, _, test_loader_for_cam, _ = build_dataloaders(DATASET_ROOT)
    test_dataset = test_loader_for_cam.dataset
    make_gradcam(trained_models[best_name], best_name, test_dataset, device)

    print("\n=== Model comparison ===")
    print(summary_df)
    print(f"\nSaved reports to: {REPORT_DIR}")




## 실행
아래 셀을 실행하면 설정된 모든 모델을 학습하고 결과 피규어를 저장한다.


In [21]:
main()


Device: cuda


  0%|          | 0/52 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

[small_cnn] epoch 001/4 | train_acc=0.6060 | val_acc=0.5352 | val_f1=0.4877


  0%|          | 0/52 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

[efficientnet_b0] epoch 003/4 | train_acc=0.9909 | val_acc=1.0000 | val_f1=1.0000


  0%|          | 0/52 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

[efficientnet_b0] epoch 004/4 | train_acc=0.9976 | val_acc=1.0000 | val_f1=1.0000

=== Model comparison ===
                model  final_train_acc  best_train_acc  best_val_acc  \
0           small_cnn         0.989124        0.989124      0.997183   
1    resnet18_scratch         0.997583        0.997583      1.000000   
2   resnet18_finetune         0.999396        0.999396      1.000000   
3     efficientnet_b0         0.997583        0.997583      1.000000   
4  mobilenet_v3_small         0.996375        0.998187      0.819718   

   test_acc  test_macro_f1  train_time_sec  sec_per_epoch  num_epochs  
0  1.000000       1.000000       33.792141       8.448035           4  
1  1.000000       1.000000       35.358107       8.839527           4  
2  1.000000       1.000000       33.688424       8.422106           4  
3  0.994366       0.994325       44.636288      11.159072           4  
4  0.833803       0.799369       29.529059       7.382265           4  

Saved reports to: cwru_mod